# BugRisk AI - Phase 7 ablation on a Colab GPU

Runs the offline research pipeline end to end and saves the results to Google Drive:

1. Download ApacheJIT and preprocess it.
2. Extract the changed source files of five Apache projects.
3. Embed them with CodeBERT (GPU).
4. Build GRU commit histories and changed-file import graphs.
5. Train the ablation: XGBoost (metrics), MLP metrics, + CodeBERT, + GRU, + GNN.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.

Start with `QUICK = True` (about 300 commits per project, roughly 20-40 minutes) to check
everything works. Then set `QUICK = False` for the full run. Every slow step is cached on
Drive, so if Colab disconnects, just run all cells again and it continues where it stopped.

When it finishes, download `ablation.json` from the Drive folder printed at the end and put it
in `ml/artifacts/advanced/` in the repository.

In [ ]:
QUICK = True  # set to False for the full five-project run

REPO_URL = "https://github.com/Sahilpatil009/BugRisk-AI.git"
BRANCH = "cursor/phase-7-advanced-model-f598"  # use "main" after the Phase 7 PR is merged
PROJECTS = "apache/zookeeper,apache/spark,apache/hadoop-mapreduce,apache/hadoop-hdfs,apache/kafka"
SEEDS = "42,43,44"

if QUICK:
    EXTRACT_ARGS = "--max-commits-per-project 300 --max-files 4 --max-chars 8000"
    EMBED_ARGS = "--batch-size 32 --max-length 256"
else:
    EXTRACT_ARGS = "--max-files 8 --max-chars 20000"
    EMBED_ARGS = "--batch-size 32 --max-length 512"
MODE = "quick" if QUICK else "full"
print("Mode:", MODE)

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE = f"/content/drive/MyDrive/bugrisk-phase7/{MODE}"
WORK = f"/content/bugrisk-run/{MODE}"
DATA_CACHE = "/content/drive/MyDrive/bugrisk-phase7/data"
for path in (DRIVE, WORK, DATA_CACHE):
    !mkdir -p "{path}"

In [ ]:
%cd /content
!test -d BugRisk-AI || git clone --quiet "{REPO_URL}" BugRisk-AI
%cd /content/BugRisk-AI
!git fetch --quiet origin "{BRANCH}" && git checkout --quiet "{BRANCH}" && git pull --quiet --ff-only origin "{BRANCH}"
!git log --oneline -1
# Colab already ships a CUDA build of torch, which satisfies the codebert extra.
!pip install --quiet -e "./backend[ml,codebert]"

## 1. ApacheJIT dataset

In [ ]:
import os

if os.path.exists(f"{DATA_CACHE}/apachejit.csv"):
    !mkdir -p ml/data/processed && cp "{DATA_CACHE}/apachejit.csv" ml/data/processed/apachejit.csv
else:
    !python scripts/fetch_apachejit.py --output ml/data/raw/apachejit.csv
    !python -m ml.preprocessing.prepare --input ml/data/raw/apachejit.csv --output ml/data/processed/apachejit.csv
    !cp ml/data/processed/apachejit.csv "{DATA_CACHE}/apachejit.csv"
!wc -l ml/data/processed/apachejit.csv

## 2. Extract changed source files

Git mirrors are cloned to the Colab disk (fast) instead of Drive (very slow for Git).
The extracted `sources.parquet` is saved to Drive.

In [ ]:
if os.path.exists(f"{DRIVE}/sources/sources.parquet"):
    print("Using cached sources from Drive")
else:
    !python -m ml.embeddings.extract --data ml/data/processed/apachejit.csv --projects "{PROJECTS}" {EXTRACT_ARGS} --cache-root /content/repos --output "{WORK}/sources"
    !mkdir -p "{DRIVE}/sources" && cp "{WORK}"/sources/* "{DRIVE}/sources/"
!cat "{DRIVE}/sources/resolution.json"

## 3. CodeBERT embeddings (GPU)

The per-file embedding cache is copied back to Drive after the step, so a rerun only encodes
files it has not seen.

In [ ]:
!mkdir -p "{WORK}/embeddings"
if os.path.exists(f"{DRIVE}/embeddings/file_cache.sqlite3"):
    !cp "{DRIVE}/embeddings/file_cache.sqlite3" "{WORK}/embeddings/"
!python -m ml.embeddings.embed --sources "{DRIVE}/sources/sources.parquet" --output "{WORK}/embeddings" --device cuda {EMBED_ARGS}
!mkdir -p "{DRIVE}/embeddings" && cp "{WORK}"/embeddings/* "{DRIVE}/embeddings/"
!cat "{DRIVE}/embeddings/embedding_meta.json"

## 4. GRU histories and dependency graphs

In [ ]:
!python -m ml.advanced.prepare --data ml/data/processed/apachejit.csv --sources "{DRIVE}/sources/sources.parquet" --output "{WORK}/advanced"
!mkdir -p "{DRIVE}/advanced" && cp "{WORK}"/advanced/* "{DRIVE}/advanced/"

## 5. Train the ablation

In [ ]:
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{WORK}/mlflow.db"
!python -m ml.training.train_advanced --data ml/data/processed/apachejit.csv --embeddings "{DRIVE}/embeddings/embeddings.npz" --inputs "{DRIVE}/advanced" --output "{WORK}/artifacts" --seeds "{SEEDS}" --device cuda
!mkdir -p "{DRIVE}/artifacts" && cp "{WORK}"/artifacts/* "{DRIVE}/artifacts/" && cp "{WORK}/mlflow.db" "{DRIVE}/artifacts/"

## 6. Results

In [ ]:
import json

import pandas as pd

with open(f"{DRIVE}/artifacts/ablation.json", encoding="utf-8") as handle:
    result = json.load(handle)
rows = []
for variant, block in result["variants"].items():
    row = {"variant": variant}
    for metric in ("precision", "recall", "f1", "f2", "roc_auc", "pr_auc"):
        stats = block["summary"][metric]
        row[metric] = f"{stats['mean']:.3f} +/- {stats['std']:.3f}"
    rows.append(row)
print("Commits:", result["subset"]["commits"], "| buggy rate:", round(result["subset"]["buggy_rate"], 3))
print("Best on validation:", result["best"])
pd.DataFrame(rows).set_index("variant")

In [ ]:
print("Download this file and put it in ml/artifacts/advanced/ in the repository:")
print(f"{DRIVE}/artifacts/ablation.json")